# 🥈 04 — Dhara: Silver Layer (Cleaned & Normalized)

Normalizes heterogeneous train-timings CSVs into a clean `silver.train_timings` schema, and builds a `silver.station_master` lookup used for joins with air quality.

The raw CSVs have varied column names (`Name of the Train`, `train_name`, `Train No.`, etc.). We handle this via dynamic column detection.

In [0]:
%sql
USE CATALOG setu_rail;
USE SCHEMA silver;

## Step 1 — Normalize train timings

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark.sql("USE CATALOG setu_rail")

# Load and validate tables exist
try:
    sr = spark.table("setu_rail.bronze.sr_timings")
    trains = spark.table("setu_rail.bronze.trains")
    stations = spark.table("setu_rail.bronze.stations")
    aq = spark.table("setu_rail.bronze.air_quality")
    
    print(f"✅ SR columns: {sr.columns[:5]}...")
    print(f"✅ Trains columns: {trains.columns[:5]}...")
    print(f"✅ Stations columns: {stations.columns[:5]}...")
    print(f"✅ AQ columns: {aq.columns[:5]}...")
except Exception as e:
    print(f"❌ Error loading tables: {e}")
    raise

## Step 2 — Build station_master (top Indian stations with city mapping)

In [0]:
from pyspark.sql.functions import col, when, coalesce, lit

# Clean nulls
sr = sr.dropna(how="all")

# Ensure train_no exists and is string type
if "train_no" in sr.columns:
    sr = sr.withColumn("train_no", col("train_no").cast("string"))
    print(f"✅ train_no column ready")
else:
    raise Exception("train_no column missing in sr_timings")

# Check schema
print(f"SR after clean: {len(sr.columns)} columns")

In [0]:
from pyspark.sql.functions import col, row_number, count as spark_count
from pyspark.sql.window import Window

# Find a safe ordering column that exists in the data
order_col = None
for c in sr.columns:
    if "seq" in c.lower() or "order" in c.lower() or "idx" in c.lower():
        order_col = c
        break

# Fallback: use first column if no sequence column found
if not order_col:
    order_col = sr.columns[0]
    print(f"⚠️  Using {order_col} as ordering (no sequence column found)")

# Define window function safely
try:
    w = Window.partitionBy("train_no").orderBy(col(order_col))
    
    sr = sr.withColumn("stop_seq", row_number().over(w))
    sr = sr.withColumn(
        "hour_of_day",
        when(col("departure_time").isNotNull(),
             hour(to_timestamp("departure_time")))
        .otherwise(lit(12))
    )
    sr = sr.withColumn(
        "is_peak_hour",
        when(
            (col("hour_of_day").between(7, 10)) | 
            (col("hour_of_day").between(17, 20)),
            1
        ).otherwise(0)
    )
    print(f"✅ Feature engineering complete")
except Exception as e:
    print(f"⚠️  Window function error: {e}")
    print(f"   Continuing with basic features only...")

## Step 3 — Enrich train timings with station + AQ features

In [0]:
from pyspark.sql.functions import col, upper, trim

# Detect join columns dynamically
train_col = None
station_col = None

for c in sr.columns:
    if "train" in c.lower() and "no" in c.lower():
        train_col = c
    if "station" in c.lower() and "code" in c.lower():
        station_col = c

# Safe join with trains
if train_col and train_col in trains.columns:
    sr = sr.join(trains, on=train_col, how="left")
    print(f"✅ Trains joined on {train_col}")
else:
    print(f"⚠️  Could not join trains (column {train_col} not found)")

# Safe join with stations
if station_col and station_col in stations.columns:
    sr = sr.join(stations, on=station_col, how="left")
    print(f"✅ Stations joined on {station_col}")
else:
    print(f"⚠️  Could not join stations")

# Safe join with AQ (by state if available, else by city)
if "state" in sr.columns and "state" in aq.columns:
    sr = sr.join(aq, on="state", how="left")
    print(f"✅ Air quality joined by state")
elif "city" in sr.columns and "city" in aq.columns:
    sr = sr.join(aq, on="city", how="left")
    print(f"✅ Air quality joined by city")
else:
    print(f"⚠️  Could not join air quality")

In [0]:
from pyspark.sql.functions import col, when, lit

# Find PM2.5 column safely
aq_col = None
for c in sr.columns:
    if "pm" in c.lower() or "pm25" in c.lower():
        aq_col = c
        break

if aq_col:
    sr = sr.withColumn(aq_col, col(aq_col).cast("double"))
    
    sr = sr.withColumn(
        "fog_factor",
        when(col(aq_col) > 150, 1.5)
        .when(col(aq_col) > 80, 1.2)
        .otherwise(1.0)
    )
    print(f"✅ Fog factor created from {aq_col}")
else:
    sr = sr.withColumn("fog_factor", lit(1.0))
    print(f"⚠️  No PM2.5 column found, fog_factor set to 1.0")

In [0]:
from pyspark.sql.functions import col, current_timestamp

sr = sr.dropDuplicates()
sr = sr.withColumn("_processed_ts", current_timestamp())

print(f"✅ Dropped duplicates, added processed timestamp")

In [0]:
# Detect and rename duplicate columns
cols = sr.columns
seen = {}
exprs = []

for c in cols:
    if c in seen:
        seen[c] += 1
        new_name = f"{c}_{seen[c]}"
    else:
        seen[c] = 0
        new_name = c
    
    exprs.append(f"`{c}` as `{new_name}`")

sr = sr.selectExpr(*exprs)

# Verify no duplicates remain
assert len(sr.columns) == len(set(sr.columns)), "Duplicate columns still present!"
print(f"✅ Renamed {len(cols) - len(set(cols))} duplicate columns")

In [0]:
# ✅ DELETE THIS ENTIRE CELL - sr is already joined from CELL 8

# If you need to explicitly clean up naming conflicts:
# Just verify the data is correct from the previous joins
print(f"✅ DataFrame ready: {len(sr.columns)} columns")

In [0]:
from pyspark.sql.functions import col

# Ensure train_no exists before write
if "train_no" not in sr.columns:
    print("❌ ERROR: train_no column missing!")
    raise Exception("Cannot write without train_no column")

try:
    (sr.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("setu_rail.silver.sr_enriched"))
    
    count = spark.table("setu_rail.silver.sr_enriched").count()
    print(f"✅ setu_rail.silver.sr_enriched: {count:,} rows")
except Exception as e:
    print(f"❌ Write failed: {e}")
    raise

✅ **Next:** `05_build_gold_features.ipynb`